# QLoRA — teach a small model the Hansard citation format


## 1. Check the GPU


In [2]:
!nvidia-smi

Sat Jul 18 17:32:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Unsloth


In [3]:
!pip install -q unsloth

## 3. Load Qwen2.5-1.5B in 4-bit and attach LoRA adapters


In [4]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.3 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


## 4. Load the dataset and render it with the chat template
Each row is `{system, prompt, completion}`. We train the model to produce the completion.


In [5]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": "train.jsonl", "val": "val.jsonl"})

def to_text(ex):
    messages = [
        {"role": "system", "content": ex["system"]},
        {"role": "user", "content": ex["prompt"]},
        {"role": "assistant", "content": ex["completion"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

ds = ds.map(to_text, remove_columns=ds["train"].column_names)
print(ds)
print(ds["train"][0]["text"][:600])


Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/108 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 108
    })
    val: Dataset({
        features: ['text'],
        num_rows: 12
    })
})
<|im_start|>system
You are a parliamentary research assistant that summarises Hansard debates. The source text is in Malay, but you MUST write your ENTIRE answer in English only — do not use any other language (keep proper nouns such as names and places as-is). Summarise each issue in your own words, but always keep the speaker's exact name and constituency. Always cite sources with [n] bracket notation.<|im_end|>
<|im_start|>user
Summarise the following Malaysian Parliament speeches IN ENGLISH.

Output ONLY a numbered list. Each line MUST be:
N. **3-5 word title**: Speaker (Constituency) one 


## 5. Train (loss on the answer only)


In [6]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

trainer.train()


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/108 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/12 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/108 [00:00<?, ? examples/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 108 | Num Epochs = 3 | Total steps = 42
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,0.719668
10,0.532944
15,0.563621
20,0.439260
25,0.419191
30,0.374752
35,0.359632
40,0.318517


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-42/tokenizer_config.json.


TrainOutput(global_step=42, training_loss=0.45916903018951416, metrics={'train_runtime': 550.1638, 'train_samples_per_second': 0.589, 'train_steps_per_second': 0.076, 'total_flos': 5912445107410944.0, 'train_loss': 0.45916903018951416, 'epoch': 3.0})

## 6. Quick check — does it produce the format natively?


In [7]:
FastLanguageModel.for_inference(model)

ex = load_dataset("json", data_files="val.jsonl")["train"][0]
prompt = tokenizer.apply_chat_template(
    [{"role": "system", "content": ex["system"]},
     {"role": "user", "content": ex["prompt"]}],
    tokenize=False, add_generation_prompt=True,
)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=300, temperature=0.3)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))


Generating train split: 0 examples [00:00, ? examples/s]

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

1. **Rang Undang-Undang Perbekalan 2025**: Datuk Seri Dr. Noraini binti Ahmad (Timbalan Menteri Pembangunan Wanita, Keluarga dan Masyarakat) supported the proposal for Rang Undang-Undang Perbekalan 2025 [2].
2. **Rang Undang-Undang Pembangunan 2025**: Tuan M. Kulasegaran a/l Murugeson (Reformasi Institusi) requested that the discussion on Rang Undang-Undang Pembangunan 2025 be postponed until it is fully discussed and decided upon [4].


## 7. Export to GGUF (q4_K_M) for Ollama
This builds llama.cpp and can take a few minutes.


In [8]:
model.save_pretrained_gguf("hansard-qwen-gguf", tokenizer, quantization_method="q4_k_m")


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in hansard-qwen-gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [03:57<00:00, 237.52s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:54<00:00, 114.54s/it]


Unsloth: Merge process complete. Saved to `/content/hansard-qwen-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10043-mix-0ac9dfb (app-b10043-mix-0ac9dfb-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['hansard-qwen-gguf_gguf/qwen2.5-1.5b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions comp

{'save_directory': 'hansard-qwen-gguf',
 'gguf_directory': 'hansard-qwen-gguf_gguf',
 'gguf_files': ['hansard-qwen-gguf_gguf/qwen2.5-1.5b-instruct.Q4_K_M.gguf'],
 'modelfile_location': 'hansard-qwen-gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

## 8. Download + deploy locally

Zip and download the GGUF:
```python
!zip -r hansard-qwen-gguf.zip hansard-qwen-gguf
from google.colab import files; files.download('hansard-qwen-gguf.zip')
```

On your laptop, unzip it, make a `Modelfile`:
```
FROM ./hansard-qwen-gguf/unsloth.Q4_K_M.gguf
PARAMETER temperature 0.3
```
```bash
ollama create hansard-qwen -f Modelfile
CUDA_VISIBLE_DEVICES="" uv run python finetune/evaluate.py qwen2.5:1.5b hansard-qwen
```


### Download cell (run to grab the GGUF)


In [9]:
!zip -r -q hansard-qwen-gguf.zip hansard-qwen-gguf
from google.colab import files
files.download("hansard-qwen-gguf.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>